In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("../dataset/dataset_limpio.csv")


region = "West"

# Preparación de los datos

datos = df[df["Region"] == region].sort_values("Year")

x = datos["Year"].tolist()
y = datos["Median AQI"].tolist()

n = len(x) - 1


# Spline cúbico Natural

def splineNatural(n: int, x: list, a: list):
    """
    Realiza el método de interpolación por splines naturales
    para un conjunto de puntos iniciales
    - Los datos de tipo lista se ingresan como una lista en la variable respectiva

    # Entradas:
        :n: número de intervalos (int)
        :x: lista de los puntos en x (list)
        :a: lista de valores de la función evaluada en cada punto en x (list)

    # Salida:
        - lista de tuplas con los coeficientes de cada polinomio del spline

    # Ejemplo:
        x = [0,1,2]
        a = [-5,-4,3]
    - Entradas:2, x, a
    - Salida:[(-5, -0.5, 0.0, 1.5), (-4, 4.0, 4.5, -1.5)]
    """

    h = [0] * n

    i = 0
    while i < n:
        h[i] = x[i+1] - x[i]
        i += 1

    alpha = [0] * n

    j = 1
    while j < n:
        alpha[j] = ((3 / h[j]) * (a[j+1] - a[j])) - \
                   ((3 / h[j-1]) * (a[j] - a[j-1]))
        j += 1

    l = [0] * (n + 1)
    u = [0] * (n + 1)
    z = [0] * (n + 1)

    l[0] = 1
    u[0] = 0
    z[0] = 0

    j = 1
    while j < n:

        l[j] = 2 * (x[j+1] - x[j-1]) - h[j-1] * u[j-1]
        u[j] = h[j] / l[j]
        z[j] = (alpha[j] - h[j-1] * z[j-1]) / l[j]
        j += 1

    b = [0] * n
    c = [0] * (n + 1)
    d = [0] * n

    l[n] = 1
    z[n] = 0
    c[n] = 0

    j = n - 1

    while j >= 0:

        c[j] = z[j] - u[j] * c[j+1]
        b[j] = ((a[j+1] - a[j]) / h[j]) - (h[j] * (c[j+1] + 2 * c[j]) / 3)
        d[j] = (c[j+1] - c[j]) / (3 * h[j])
        j -= 1

    resultado = []

    j = 0
    while j < n:

        resultado.append((a[j], b[j], c[j], d[j]))
        j += 1

    return resultado

coeficientes = splineNatural(n, x, y)

# Evaluación del spline

def evaluarSpline(x_eval: float, x: list, coeficientes: list):
    """
    Evalúa el spline cúbico natural en un punto específico
    - Los datos de tipo lista se ingresan como una lista en la variable respectiva

    # Entradas:
        :x_eval: punto en el que se desea evaluar el spline (float)
        :x: lista de los puntos en x (list)
        :coeficientes: lista de tuplas con los coeficientes de cada polinomio del spline (list)
    # Salida:
        - valor del spline evaluado en x_eval (float)
    # Ejemplo:
        x_eval = 1.5
        x = [0,1,2]
        coeficientes = [(-5, -0.5, 0.0, 1.5), (-4, 4.0, 4.5, -1.5)]
    - Entradas:1.5, x, coeficientes
    - Salida:0.625
    """

    n = len(coeficientes)

    for i in range(n):
        if x[i] <= x_eval <= x[i+1]:
            a, b, c, d = coeficientes[i]
            dx = x_eval - x[i]
            return a + b * dx + c * (dx**2) + d * (dx**3)

    raise ValueError("El punto de evaluación está fuera del rango de los datos")    

x_suavisado = np.linspace(min(x), max(x), 500)

y_suavisado = []

for xi in x_suavisado:
    y_suavisado.append(evaluarSpline(xi, x, coeficientes))



# Notas

- Modificar para permitir la elección de la region a analizar
- Implementar metodo que grafique los datos